In [2]:
# ======================================================================
# SHORT ANSWER VALIDATION — FINAL SCIENTIFIC STATISTICAL ANALYSIS
# ======================================================================
#
# Inputs
# ------
# short_answers_merged_not_yet_adjudicated.xlsx
# short_answer_consolidated_adjudicated.xlsx
#
# Outputs
# -------
# short_answer_validation_FINAL_metrics.xlsx
# short_answer_validation_FINAL_paper_summary.txt
#
# Important methodological choices
# --------------------------------
# 1. Main inter-annotator reliability is computed on all double-reviewed
#    items having valid labels from two distinct validators.
#
# 2. "Strict complete" pairs (supporting source present, required comment
#    present, etc.) are analyzed separately as a sensitivity analysis.
#
# 3. Since the physical validators differ across items, the main report
#    includes Fleiss' kappa and Krippendorff's alpha in addition to
#    pooled primary-vs-second Cohen's kappa.
#
# 4. Accepted variants are analyzed explicitly from:
#       answer
#       primary_accepted_variants
#       second_accepted_variants
#       merged_accepted_variants
#
#    Pipe-separated variants are parsed as sets.
#
# 5. final_decision == "accept_with_variants" is NOT used to infer
#    whether an item has accepted variants.
#
# ======================================================================

from pathlib import Path
import math
import re
import unicodedata

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter


# ======================================================================
# CONFIGURATION
# ======================================================================

PRE_FILE = Path("short_answers_merged_not_yet_adjudicated.xlsx")
POST_FILE = Path("short_answer_consolidated_adjudicated.xlsx")

OUTPUT_XLSX = Path("short_answer_validation_FINAL_metrics.xlsx")
OUTPUT_TXT = Path("short_answer_validation_FINAL_paper_summary.txt")

BOOTSTRAP_REPS = 5000
RANDOM_SEED = 20260812

# Do not treat a blank adjudicator_id as an error.
REQUIRE_ADJUDICATOR_ID = False

# How to interpret adjudicated_accepted_variants when the adjudicator filled it.
#
# "replace_if_filled":
#     adjudicated_accepted_variants is considered the final authoritative set.
#
# "union":
#     adjudicated_accepted_variants is added to merged_accepted_variants.
#
# I recommend replace_if_filled given the name of your column.
ADJUDICATED_VARIANTS_POLICY = "replace_if_filled"

ANSWER_CATEGORIES = [
    "correct",
    "minor_fix",
    "incorrect",
]

QUESTION_CATEGORIES = [
    "ok",
    "review",
]

FINAL_DECISIONS = [
    "accept",
    "accept_with_variants",
    "corrected_and_accept",
    "rewrite_question",
    "reject_item",
]

RETAIN_DECISIONS = {
    "accept",
    "accept_with_variants",
    "corrected_and_accept",
}


# ======================================================================
# BASIC UTILITIES
# ======================================================================

def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def has_value(value):
    return clean_text(value) != ""


def normalize_item_id(value):
    if pd.isna(value):
        return ""

    if isinstance(value, (int, np.integer)):
        return str(int(value))

    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return str(int(value))

    return str(value).strip()


def normalize_label(value):
    text = clean_text(value).lower()
    text = re.sub(r"[\s\-/]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


ARABIC_DIACRITICS = re.compile(
    r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]"
)


def normalize_variant(value):
    """
    Conservative normalization used only for SET COMPARISON.

    Original text is never modified in the dataset.

    Handles:
    - Unicode normalization
    - Arabic tashkil
    - tatweel
    - case differences in Latin text
    - repeated spaces
    """
    text = clean_text(value)

    if not text:
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = ARABIC_DIACRITICS.sub("", text)
    text = text.replace("ـ", "")
    text = text.casefold()
    text = re.sub(r"\s+", " ", text).strip()

    return text


def parse_pipe_variants(value):
    """
    Example:
        مدينة الرباط | Rabat | الرباط عاصمة المغرب

    -> three distinct variants.
    """
    text = clean_text(value)

    if not text:
        return []

    result = []
    seen = set()

    for part in text.split("|"):
        part = clean_text(part)

        if not part:
            continue

        key = normalize_variant(part)

        if key and key not in seen:
            seen.add(key)
            result.append(part)

    return result


def variant_normalized_set(value):
    return {
        normalize_variant(v)
        for v in parse_pipe_variants(value)
        if normalize_variant(v)
    }


def safe_percentage(numerator, denominator):
    if denominator == 0:
        return np.nan
    return 100 * numerator / denominator


def fmt(value, decimals=3):
    if value is None or not np.isfinite(value):
        return "NA"
    return f"{value:.{decimals}f}"


def fmt_pct(value, decimals=1):
    if value is None or not np.isfinite(value):
        return "NA"
    return f"{value:.{decimals}f}%"


def numeric_column(df, column):
    if column not in df.columns:
        return pd.Series(0, index=df.index, dtype=float)

    return pd.to_numeric(
        df[column],
        errors="coerce",
    ).fillna(0)


# ======================================================================
# LOAD EXCEL
# ======================================================================

def read_sheet(path, sheet_name, required=True):

    if not path.exists():
        raise FileNotFoundError(path)

    xls = pd.ExcelFile(
        path,
        engine="openpyxl",
    )

    if sheet_name not in xls.sheet_names:
        if required:
            raise ValueError(
                f"{sheet_name!r} absent de {path.name}. "
                f"Feuilles disponibles: {xls.sheet_names}"
            )

        return pd.DataFrame()

    df = pd.read_excel(
        path,
        sheet_name=sheet_name,
        dtype=object,
        engine="openpyxl",
    )

    if "item_id" in df.columns:
        df["item_id"] = (
            df["item_id"]
            .map(normalize_item_id)
        )

    return df


pre = read_sheet(
    PRE_FILE,
    "CONSOLIDATED_ITEMS",
)

pre_queue = read_sheet(
    PRE_FILE,
    "ADJUDICATION_QUEUE",
)

judgments = read_sheet(
    PRE_FILE,
    "ALL_JUDGMENTS",
    required=False,
)

post_queue = read_sheet(
    POST_FILE,
    "ADJUDICATION_QUEUE",
)


# ======================================================================
# REQUIRED COLUMN CHECK
# ======================================================================

required = [
    "item_id",
    "answer",

    "primary_annotator_id",
    "primary_answer_correctness",
    "primary_question_quality",
    "primary_accepted_variants",
    "primary_judgment_complete",

    "second_annotator_id",
    "second_answer_correctness",
    "second_question_quality",
    "second_accepted_variants",
    "second_judgment_complete",

    "merged_accepted_variants",

    "primary_judgment_count",
    "second_check_judgment_count",

    "provisional_decision",
    "requires_adjudication",
]

missing = [
    c for c in required
    if c not in pre.columns
]

if missing:
    raise KeyError(
        "Missing CONSOLIDATED_ITEMS columns:\n"
        + "\n".join(missing)
    )


# ======================================================================
# RECONSTRUCT POST-ADJUDICATION TABLE
# ======================================================================

adjudication_columns = [
    "adjudicator_id",
    "adjudication_answer_correctness",
    "adjudication_question_quality",
    "adjudicated_answer",
    "adjudicated_accepted_variants",
    "adjudication_comment",
    "final_decision",
    "adjudication_status",
]

for col in adjudication_columns:
    if col not in post_queue.columns:
        raise KeyError(
            f"Missing adjudication column: {col}"
        )


pre_queue_base = pre_queue.drop(
    columns=adjudication_columns,
    errors="ignore",
)

post_adj = post_queue[
    ["item_id"] + adjudication_columns
].copy()

adj = pre_queue_base.merge(
    post_adj,
    on="item_id",
    how="outer",
    validate="one_to_one",
)

for col in [
    "adjudication_answer_correctness",
    "adjudication_question_quality",
    "final_decision",
    "adjudication_status",
]:
    adj[col] = adj[col].map(
        normalize_label
    )


# ======================================================================
# DOUBLE-REVIEW BASE
# ======================================================================

double_mask = (
    (numeric_column(pre, "primary_judgment_count") == 1)
    & (numeric_column(pre, "second_check_judgment_count") == 1)
    & pre["primary_annotator_id"].map(has_value)
    & pre["second_annotator_id"].map(has_value)
    & (
        pre["primary_annotator_id"].map(clean_text)
        != pre["second_annotator_id"].map(clean_text)
    )
)

double_review = pre.loc[
    double_mask
].copy()


# ======================================================================
# IMPORTANT CHANGE:
# LABEL-VALID PAIRS != STRICT COMPLETE PAIRS
# ======================================================================

answer_valid_mask = (
    double_review[
        "primary_answer_correctness"
    ]
    .map(normalize_label)
    .isin(ANSWER_CATEGORIES)
    &
    double_review[
        "second_answer_correctness"
    ]
    .map(normalize_label)
    .isin(ANSWER_CATEGORIES)
)

question_valid_mask = (
    double_review[
        "primary_question_quality"
    ]
    .map(normalize_label)
    .isin(QUESTION_CATEGORIES)
    &
    double_review[
        "second_question_quality"
    ]
    .map(normalize_label)
    .isin(QUESTION_CATEGORIES)
)


answer_pairs = double_review.loc[
    answer_valid_mask
].copy()

question_pairs = double_review.loc[
    question_valid_mask
].copy()


for col in [
    "primary_answer_correctness",
    "second_answer_correctness",
    "primary_question_quality",
    "second_question_quality",
]:
    if col in answer_pairs.columns:
        answer_pairs[col] = (
            answer_pairs[col]
            .map(normalize_label)
        )

    if col in question_pairs.columns:
        question_pairs[col] = (
            question_pairs[col]
            .map(normalize_label)
        )


strict_mask = (
    (
        double_review["primary_judgment_complete"]
        .map(normalize_label)
        == "yes"
    )
    &
    (
        double_review["second_judgment_complete"]
        .map(normalize_label)
        == "yes"
    )
)

strict_double = double_review.loc[
    strict_mask
].copy()

strict_answer = strict_double.loc[
    strict_double[
        "primary_answer_correctness"
    ]
    .map(normalize_label)
    .isin(ANSWER_CATEGORIES)
    &
    strict_double[
        "second_answer_correctness"
    ]
    .map(normalize_label)
    .isin(ANSWER_CATEGORIES)
].copy()

strict_question = strict_double.loc[
    strict_double[
        "primary_question_quality"
    ]
    .map(normalize_label)
    .isin(QUESTION_CATEGORIES)
    &
    strict_double[
        "second_question_quality"
    ]
    .map(normalize_label)
    .isin(QUESTION_CATEGORIES)
].copy()


# ======================================================================
# AGREEMENT METRICS
# ======================================================================

def raw_agreement(a, b):
    a = np.asarray(a)
    b = np.asarray(b)

    if len(a) == 0:
        return np.nan

    return np.mean(a == b)


def cohen_kappa(
    a,
    b,
    categories,
    weighting=None,
):
    """
    Here Cohen's kappa is a POOLED ROLE-BASED statistic:
    primary judgments vs second-check judgments.

    Since physical validator identity varies by item,
    it should not be the only reliability statistic reported.
    """

    a = np.asarray(a, dtype=object)
    b = np.asarray(b, dtype=object)

    valid = np.array([
        x in categories and y in categories
        for x, y in zip(a, b)
    ])

    a = a[valid]
    b = b[valid]

    if len(a) == 0:
        return np.nan

    mapping = {
        c: i
        for i, c in enumerate(categories)
    }

    k = len(categories)

    table = np.zeros(
        (k, k),
        dtype=float,
    )

    for x, y in zip(a, b):
        table[
            mapping[x],
            mapping[y]
        ] += 1

    observed = table / table.sum()

    expected = np.outer(
        observed.sum(axis=1),
        observed.sum(axis=0),
    )

    if weighting is None:
        weights = (
            np.ones((k, k))
            - np.eye(k)
        )

    elif weighting == "linear":
        denominator = max(k - 1, 1)

        weights = np.fromfunction(
            lambda i, j:
                np.abs(i - j) / denominator,
            (k, k),
            dtype=float,
        )

    elif weighting == "quadratic":
        denominator = max(k - 1, 1)

        weights = np.fromfunction(
            lambda i, j:
                ((i - j) / denominator) ** 2,
            (k, k),
            dtype=float,
        )

    else:
        raise ValueError(weighting)

    Do = (
        weights * observed
    ).sum()

    De = (
        weights * expected
    ).sum()

    if De == 0:
        return (
            1.0
            if Do == 0
            else np.nan
        )

    return 1 - Do / De


def fleiss_kappa_two_ratings(
    a,
    b,
    categories,
):
    """
    Fleiss' kappa.

    Unlike Cohen's kappa, this formulation does not require the
    physical person acting as reviewer 1 / reviewer 2 to remain
    constant over all items.

    Every double-reviewed item has n=2 ratings.
    """

    a = np.asarray(a, dtype=object)
    b = np.asarray(b, dtype=object)

    valid = np.array([
        x in categories and y in categories
        for x, y in zip(a, b)
    ])

    a = a[valid]
    b = b[valid]

    N = len(a)

    if N == 0:
        return np.nan

    K = len(categories)

    counts = np.zeros(
        (N, K),
        dtype=float,
    )

    mapping = {
        c: i
        for i, c in enumerate(categories)
    }

    for i, (x, y) in enumerate(
        zip(a, b)
    ):
        counts[
            i,
            mapping[x]
        ] += 1

        counts[
            i,
            mapping[y]
        ] += 1

    n = 2

    # Item-level observed agreement
    P_i = (
        (counts ** 2).sum(axis=1)
        - n
    ) / (
        n * (n - 1)
    )

    P_bar = P_i.mean()

    # Overall category prevalence
    p_j = (
        counts.sum(axis=0)
        / (N * n)
    )

    P_e = (
        p_j ** 2
    ).sum()

    if P_e == 1:
        return (
            1.0
            if P_bar == 1
            else np.nan
        )

    return (
        P_bar - P_e
    ) / (
        1 - P_e
    )


def krippendorff_alpha_nominal(
    a,
    b,
    categories,
):
    """
    Nominal Krippendorff alpha for units with two ratings.

    Appropriate for a design in which different human validators
    can rate different items.
    """

    pairs = [
        (x, y)
        for x, y in zip(a, b)
        if x in categories
        and y in categories
    ]

    if not pairs:
        return np.nan

    # Observed disagreement
    Do = np.mean([
        0 if x == y else 1
        for x, y in pairs
    ])

    ratings = [
        r
        for pair in pairs
        for r in pair
    ]

    N = len(ratings)

    if N < 2:
        return np.nan

    counts = {
        c: ratings.count(c)
        for c in categories
    }

    observed_expected_agreement = sum(
        count * (count - 1)
        for count in counts.values()
    ) / (
        N * (N - 1)
    )

    De = (
        1
        - observed_expected_agreement
    )

    if De == 0:
        return (
            1.0
            if Do == 0
            else np.nan
        )

    return 1 - Do / De


def gwet_ac1(
    a,
    b,
    categories,
):
    """
    Gwet AC1.

    Particularly useful as a sensitivity statistic when one label
    (e.g. correct) is highly prevalent and Cohen/Fleiss kappa exhibits
    the prevalence paradox.
    """

    a = np.asarray(a, dtype=object)
    b = np.asarray(b, dtype=object)

    valid = np.array([
        x in categories and y in categories
        for x, y in zip(a, b)
    ])

    a = a[valid]
    b = b[valid]

    if len(a) == 0:
        return np.nan

    Po = np.mean(a == b)

    q = len(categories)

    p = []

    for category in categories:

        p1 = np.mean(
            a == category
        )

        p2 = np.mean(
            b == category
        )

        p.append(
            (p1 + p2) / 2
        )

    p = np.asarray(p)

    Pe = (
        np.sum(
            p * (1 - p)
        )
        / (q - 1)
    )

    if 1 - Pe == 0:
        return np.nan

    return (
        Po - Pe
    ) / (
        1 - Pe
    )


# ======================================================================
# PAIRED BOOTSTRAP CONFIDENCE INTERVAL
# ======================================================================

def bootstrap_ci(
    a,
    b,
    metric_function,
    repetitions=BOOTSTRAP_REPS,
    seed=RANDOM_SEED,
):
    a = np.asarray(
        list(a),
        dtype=object,
    )

    b = np.asarray(
        list(b),
        dtype=object,
    )

    n = len(a)

    if n < 2:
        return np.nan, np.nan

    rng = np.random.default_rng(
        seed
    )

    values = []

    for _ in range(repetitions):

        idx = rng.integers(
            0,
            n,
            size=n,
        )

        try:
            score = metric_function(
                a[idx],
                b[idx],
            )
        except Exception:
            continue

        if np.isfinite(score):
            values.append(score)

    if len(values) < 30:
        return np.nan, np.nan

    return tuple(
        np.percentile(
            values,
            [2.5, 97.5],
        )
    )


# ======================================================================
# RELIABILITY TABLE
# ======================================================================

def reliability_table(
    pair_df,
    primary_column,
    second_column,
    categories,
    dimension_name,
    ordinal=False,
    sample_name="main_label_valid",
):

    a = (
        pair_df[primary_column]
        .map(normalize_label)
        .to_numpy()
    )

    b = (
        pair_df[second_column]
        .map(normalize_label)
        .to_numpy()
    )

    metric_specs = [
        (
            "raw_agreement",
            lambda x, y:
                raw_agreement(x, y),
        ),
        (
            "fleiss_kappa",
            lambda x, y:
                fleiss_kappa_two_ratings(
                    x,
                    y,
                    categories,
                ),
        ),
        (
            "krippendorff_alpha_nominal",
            lambda x, y:
                krippendorff_alpha_nominal(
                    x,
                    y,
                    categories,
                ),
        ),
        (
            "gwet_ac1",
            lambda x, y:
                gwet_ac1(
                    x,
                    y,
                    categories,
                ),
        ),
        (
            "pooled_role_cohen_kappa_unweighted",
            lambda x, y:
                cohen_kappa(
                    x,
                    y,
                    categories,
                ),
        ),
    ]

    if ordinal:
        metric_specs.extend([
            (
                "pooled_role_cohen_kappa_linear",
                lambda x, y:
                    cohen_kappa(
                        x,
                        y,
                        categories,
                        "linear",
                    ),
            ),
            (
                "pooled_role_cohen_kappa_quadratic",
                lambda x, y:
                    cohen_kappa(
                        x,
                        y,
                        categories,
                        "quadratic",
                    ),
            ),
        ])

    rows = []

    for i, (
        metric_name,
        metric_func,
    ) in enumerate(metric_specs):

        estimate = metric_func(
            a,
            b,
        )

        low, high = bootstrap_ci(
            a,
            b,
            metric_func,
            seed=RANDOM_SEED + i * 100,
        )

        rows.append({
            "sample": sample_name,
            "dimension": dimension_name,
            "n_pairs": len(a),
            "metric": metric_name,
            "estimate": estimate,
            "ci95_low": low,
            "ci95_high": high,
        })

    return pd.DataFrame(rows)


main_answer_reliability = reliability_table(
    answer_pairs,
    "primary_answer_correctness",
    "second_answer_correctness",
    ANSWER_CATEGORIES,
    "answer_correctness",
    ordinal=True,
)

main_question_reliability = reliability_table(
    question_pairs,
    "primary_question_quality",
    "second_question_quality",
    QUESTION_CATEGORIES,
    "question_quality",
    ordinal=False,
)

strict_answer_reliability = reliability_table(
    strict_answer,
    "primary_answer_correctness",
    "second_answer_correctness",
    ANSWER_CATEGORIES,
    "answer_correctness",
    ordinal=True,
    sample_name="strict_complete_sensitivity",
)

strict_question_reliability = reliability_table(
    strict_question,
    "primary_question_quality",
    "second_question_quality",
    QUESTION_CATEGORIES,
    "question_quality",
    ordinal=False,
    sample_name="strict_complete_sensitivity",
)

reliability = pd.concat([
    main_answer_reliability,
    main_question_reliability,
    strict_answer_reliability,
    strict_question_reliability,
], ignore_index=True)


# ======================================================================
# CONFUSION MATRICES — ESSENTIAL TO INTERPRET KAPPA
# ======================================================================

answer_confusion = pd.crosstab(
    answer_pairs[
        "primary_answer_correctness"
    ].map(normalize_label),
    answer_pairs[
        "second_answer_correctness"
    ].map(normalize_label),
).reindex(
    index=ANSWER_CATEGORIES,
    columns=ANSWER_CATEGORIES,
    fill_value=0,
)

answer_confusion.index.name = "primary"
answer_confusion.columns.name = "second_check"
answer_confusion = answer_confusion.reset_index()


question_confusion = pd.crosstab(
    question_pairs[
        "primary_question_quality"
    ].map(normalize_label),
    question_pairs[
        "second_question_quality"
    ].map(normalize_label),
).reindex(
    index=QUESTION_CATEGORIES,
    columns=QUESTION_CATEGORIES,
    fill_value=0,
)

question_confusion.index.name = "primary"
question_confusion.columns.name = "second_check"
question_confusion = question_confusion.reset_index()


# ======================================================================
# MARGINAL LABEL DISTRIBUTIONS
#
# This is ESSENTIAL because it explains prevalence-related kappa behavior.
# ======================================================================

def marginal_distribution(
    pair_df,
    primary_col,
    second_col,
    categories,
    dimension,
):

    rows = []

    for role, col in [
        ("primary", primary_col),
        ("second_check", second_col),
    ]:

        values = (
            pair_df[col]
            .map(normalize_label)
        )

        n = len(values)

        for category in categories:

            count = int(
                (values == category).sum()
            )

            rows.append({
                "dimension": dimension,
                "review_role": role,
                "category": category,
                "count": count,
                "percent": safe_percentage(
                    count,
                    n,
                ),
            })

    return pd.DataFrame(rows)


marginals = pd.concat([
    marginal_distribution(
        answer_pairs,
        "primary_answer_correctness",
        "second_answer_correctness",
        ANSWER_CATEGORIES,
        "answer_correctness",
    ),
    marginal_distribution(
        question_pairs,
        "primary_question_quality",
        "second_question_quality",
        QUESTION_CATEGORIES,
        "question_quality",
    ),
], ignore_index=True)


# ======================================================================
# ACCEPTED VARIANTS ANALYSIS
# ======================================================================

variant_rows = []

for _, row in pre.iterrows():

    primary_set = variant_normalized_set(
        row.get(
            "primary_accepted_variants",
            "",
        )
    )

    second_set = variant_normalized_set(
        row.get(
            "second_accepted_variants",
            "",
        )
    )

    merged_set = variant_normalized_set(
        row.get(
            "merged_accepted_variants",
            "",
        )
    )

    reference_answer = normalize_variant(
        row.get(
            "answer",
            "",
        )
    )

    union_set = (
        primary_set
        | second_set
    )

    at_least_one_reviewer_variant = bool(
        primary_set
        or second_set
    )

    if (
        primary_set
        or second_set
    ):
        jaccard = (
            len(
                primary_set
                & second_set
            )
            /
            len(
                primary_set
                | second_set
            )
        )
    else:
        jaccard = np.nan

    full_answer_inventory = set(
        merged_set
    )

    if reference_answer:
        full_answer_inventory.add(
            reference_answer
        )

    variant_rows.append({
        "item_id":
            row["item_id"],

        "answer":
            row["answer"],

        "primary_variant_count":
            len(primary_set),

        "second_variant_count":
            len(second_set),

        "merged_variant_count":
            len(merged_set),

        "primary_has_variants":
            bool(primary_set),

        "second_has_variants":
            bool(second_set),

        "merged_has_variants":
            bool(merged_set),

        "reviewer_variant_union_count":
            len(union_set),

        "merged_equals_primary_second_union":
            merged_set == union_set,

        "second_reviewer_new_variant_count":
            len(
                second_set
                - primary_set
            ),

        "primary_reviewer_unique_variant_count":
            len(
                primary_set
                - second_set
            ),

        "variant_set_exact_agreement":
            (
                primary_set == second_set
                if at_least_one_reviewer_variant
                else np.nan
            ),

        "variant_jaccard":
            jaccard,

        "reference_answer_repeated_inside_merged_variants":
            (
                reference_answer
                in merged_set
                if reference_answer
                else False
            ),

        # Gold-answer inventory = answer + accepted variants
        "acceptable_answer_inventory_size":
            len(
                full_answer_inventory
            ),
    })


variant_detail = pd.DataFrame(
    variant_rows
)


double_ids = set(
    double_review["item_id"]
)

double_variant_detail = (
    variant_detail[
        variant_detail["item_id"]
        .isin(double_ids)
    ]
    .copy()
)


at_least_one_variant = (
    double_variant_detail[
        "primary_has_variants"
    ]
    |
    double_variant_detail[
        "second_has_variants"
    ]
)


variant_summary = pd.DataFrame([
    {
        "metric":
            "items_total",

        "value":
            len(variant_detail),
    },
    {
        "metric":
            "items_with_primary_variants",

        "value":
            int(
                variant_detail[
                    "primary_has_variants"
                ].sum()
            ),
    },
    {
        "metric":
            "items_with_second_variants",

        "value":
            int(
                variant_detail[
                    "second_has_variants"
                ].sum()
            ),
    },
    {
        "metric":
            "items_with_merged_variants",

        "value":
            int(
                variant_detail[
                    "merged_has_variants"
                ].sum()
            ),
    },
    {
        "metric":
            "merged_variant_enrichment_rate_percent",

        "value":
            safe_percentage(
                variant_detail[
                    "merged_has_variants"
                ].sum(),
                len(variant_detail),
            ),
    },
    {
        "metric":
            "total_merged_alternative_variants",

        "value":
            int(
                variant_detail[
                    "merged_variant_count"
                ].sum()
            ),
    },
    {
        "metric":
            "mean_merged_variants_per_item",

        "value":
            variant_detail[
                "merged_variant_count"
            ].mean(),
    },
    {
        "metric":
            "median_merged_variants_per_item",

        "value":
            variant_detail[
                "merged_variant_count"
            ].median(),
    },
    {
        "metric":
            "mean_acceptable_answer_inventory_size",

        "value":
            variant_detail[
                "acceptable_answer_inventory_size"
            ].mean(),
    },
    {
        "metric":
            "double_review_items_where_at_least_one_reviewer_added_variants",

        "value":
            int(
                at_least_one_variant.sum()
            ),
    },
    {
        "metric":
            "double_review_items_where_second_reviewer_added_new_variants",

        "value":
            int(
                (
                    double_variant_detail[
                        "second_reviewer_new_variant_count"
                    ]
                    > 0
                ).sum()
            ),
    },
    {
        "metric":
            "mean_variant_jaccard_when_at_least_one_reviewer_proposed_variants",

        "value":
            double_variant_detail.loc[
                at_least_one_variant,
                "variant_jaccard",
            ].mean(),
    },
    {
        "metric":
            "merged_union_consistency_rate_percent",

        "value":
            safe_percentage(
                variant_detail[
                    "merged_equals_primary_second_union"
                ].sum(),
                len(variant_detail),
            ),
    },
])


# ======================================================================
# IMPORTANT:
# Variant Jaccard is NOT an inter-rater reliability statistic.
#
# Validators are enriching an answer set; one may discover an additional
# valid synonym that the other does not. Low Jaccard does not imply a bad
# annotation. It is an enrichment/descriptive statistic only.
# ======================================================================


# ======================================================================
# STRATIFIED SECOND-REVIEW REPRESENTATIVENESS
# ======================================================================

full_distribution = (
    pre.groupby(
        ["Axe", "Difficulty"],
        dropna=False,
    )
    .size()
    .rename("full_n")
    .reset_index()
)

second_distribution = (
    double_review.groupby(
        ["Axe", "Difficulty"],
        dropna=False,
    )
    .size()
    .rename("double_review_n")
    .reset_index()
)

sampling_rep = full_distribution.merge(
    second_distribution,
    on=[
        "Axe",
        "Difficulty",
    ],
    how="left",
)

sampling_rep[
    "double_review_n"
] = (
    sampling_rep[
        "double_review_n"
    ]
    .fillna(0)
    .astype(int)
)

sampling_rep[
    "full_percent"
] = (
    100
    * sampling_rep["full_n"]
    / sampling_rep["full_n"].sum()
)

sampling_rep[
    "double_review_percent"
] = (
    100
    * sampling_rep["double_review_n"]
    / sampling_rep["double_review_n"].sum()
)

sampling_rep[
    "difference_percentage_points"
] = (
    sampling_rep[
        "double_review_percent"
    ]
    -
    sampling_rep[
        "full_percent"
    ]
)

sampling_tvd = (
    0.5
    * np.abs(
        sampling_rep[
            "full_percent"
        ] / 100
        -
        sampling_rep[
            "double_review_percent"
        ] / 100
    ).sum()
)


# ======================================================================
# ADJUDICATION ANALYSIS
# ======================================================================

adj["resolved"] = (
    (
        adj["adjudication_status"]
        == "completed"
    )
    &
    adj["final_decision"]
    .isin(FINAL_DECISIONS)
)

resolved = adj[
    adj["resolved"]
].copy()


final_decision_distribution = (
    resolved[
        "final_decision"
    ]
    .value_counts()
    .reindex(
        FINAL_DECISIONS,
        fill_value=0,
    )
    .rename_axis(
        "final_decision"
    )
    .reset_index(
        name="count"
    )
)

final_decision_distribution[
    "percent_of_resolved_adjudications"
] = (
    100
    * final_decision_distribution["count"]
    / max(
        len(resolved),
        1,
    )
)

final_decision_distribution[
    "percent_of_all_items"
] = (
    100
    * final_decision_distribution["count"]
    / len(pre)
)


# ======================================================================
# PRELIMINARY -> FINAL DECISION TRANSITIONS
# ======================================================================

transition_matrix = pd.crosstab(
    resolved[
        "provisional_decision"
    ],
    resolved[
        "final_decision"
    ],
).reindex(
    columns=FINAL_DECISIONS,
    fill_value=0,
)

transition_matrix.index.name = (
    "provisional_decision"
)

transition_matrix = (
    transition_matrix
    .reset_index()
)


# ======================================================================
# BUILD FINAL RETAINED ANSWER INVENTORY
# ======================================================================

adj_for_merge = adj[
    [
        "item_id",
        "adjudicated_answer",
        "adjudicated_accepted_variants",
        "final_decision",
        "adjudication_status",
    ]
].copy()


final_items = pre.merge(
    adj_for_merge,
    on="item_id",
    how="left",
)

final_rows = []

for _, row in final_items.iterrows():

    requires_adj = (
        normalize_label(
            row.get(
                "requires_adjudication",
                "",
            )
        )
        == "yes"
    )

    final_decision = normalize_label(
        row.get(
            "final_decision",
            "",
        )
    )

    if not requires_adj:
        retain = True
        status = "automatic_accept"

    else:
        retain = (
            final_decision
            in RETAIN_DECISIONS
        )
        status = final_decision

    if not retain:
        continue

    original_answer = clean_text(
        row.get(
            "answer",
            "",
        )
    )

    adjudicated_answer = clean_text(
        row.get(
            "adjudicated_answer",
            "",
        )
    )

    if (
        final_decision
        == "corrected_and_accept"
        and adjudicated_answer
    ):
        final_answer = adjudicated_answer
    else:
        final_answer = original_answer

    merged_variants = (
        variant_normalized_set(
            row.get(
                "merged_accepted_variants",
                "",
            )
        )
    )

    adj_variants = (
        variant_normalized_set(
            row.get(
                "adjudicated_accepted_variants",
                "",
            )
        )
    )

    if adj_variants:

        if (
            ADJUDICATED_VARIANTS_POLICY
            == "replace_if_filled"
        ):
            final_variants = adj_variants

        elif (
            ADJUDICATED_VARIANTS_POLICY
            == "union"
        ):
            final_variants = (
                merged_variants
                | adj_variants
            )

        else:
            raise ValueError(
                ADJUDICATED_VARIANTS_POLICY
            )

    else:
        final_variants = (
            merged_variants
        )

    final_answer_norm = (
        normalize_variant(
            final_answer
        )
    )

    # A reference answer should not also count as an alternative variant.
    final_variants = {
        v
        for v in final_variants
        if v != final_answer_norm
    }

    final_rows.append({
        "item_id":
            row["item_id"],

        "Axe":
            row.get("Axe", ""),

        "Difficulty":
            row.get("Difficulty", ""),

        "original_answer":
            original_answer,

        "final_answer":
            final_answer,

        "final_status":
            status,

        "final_alternative_variant_count":
            len(final_variants),

        "final_acceptable_answer_inventory_size":
            1 + len(final_variants),
    })


final_retained_items = pd.DataFrame(
    final_rows
)


# ======================================================================
# DATA QUALITY — WITHOUT FALSE ADJUDICATOR-ID WARNINGS
# ======================================================================

issues = []


def add_issue(
    severity,
    issue_type,
    item_id="",
    detail="",
):
    issues.append({
        "severity": severity,
        "issue_type": issue_type,
        "item_id": item_id,
        "detail": detail,
    })


pre_queue_ids = set(
    pre_queue["item_id"]
)

post_queue_ids = set(
    post_queue["item_id"]
)


for item in sorted(
    pre_queue_ids
    - post_queue_ids
):
    add_issue(
        "error",
        "missing_post_adjudication_item",
        item,
    )


for item in sorted(
    post_queue_ids
    - pre_queue_ids
):
    add_issue(
        "error",
        "unexpected_post_adjudication_item",
        item,
    )


for _, row in resolved.iterrows():

    if (
        REQUIRE_ADJUDICATOR_ID
        and not has_value(
            row.get(
                "adjudicator_id",
                "",
            )
        )
    ):
        add_issue(
            "warning",
            "completed_without_adjudicator_id",
            row["item_id"],
        )

    if (
        row["final_decision"]
        == "corrected_and_accept"
        and not has_value(
            row.get(
                "adjudicated_answer",
                "",
            )
        )
    ):
        add_issue(
            "warning",
            "corrected_accept_without_adjudicated_answer",
            row["item_id"],
        )


# Check that merge logic actually produced union(primary, second)
for _, row in variant_detail.iterrows():

    if not row[
        "merged_equals_primary_second_union"
    ]:
        add_issue(
            "warning",
            "merged_variants_not_equal_to_primary_second_union",
            row["item_id"],
        )


if issues:
    quality_issues = pd.DataFrame(
        issues
    )
else:
    quality_issues = pd.DataFrame([{
        "severity": "info",
        "issue_type": "no_detected_structural_issue",
        "item_id": "",
        "detail": "",
    }])


# ======================================================================
# OVERVIEW
# ======================================================================

n_items = len(pre)
n_double = len(double_review)

n_answer_pairs = len(
    answer_pairs
)

n_question_pairs = len(
    question_pairs
)

n_strict_answer = len(
    strict_answer
)

n_adjudicated = len(
    pre_queue
)

n_resolved = len(
    resolved
)


decision_counts = (
    resolved[
        "final_decision"
    ]
    .value_counts()
)

n_accept = int(
    decision_counts.get(
        "accept",
        0,
    )
)

n_accept_variants = int(
    decision_counts.get(
        "accept_with_variants",
        0,
    )
)

n_corrected = int(
    decision_counts.get(
        "corrected_and_accept",
        0,
    )
)

n_rewrite = int(
    decision_counts.get(
        "rewrite_question",
        0,
    )
)

n_reject = int(
    decision_counts.get(
        "reject_item",
        0,
    )
)


overview = pd.DataFrame([
    {
        "metric":
            "total_items",
        "value":
            n_items,
    },
    {
        "metric":
            "double_reviewed_items",
        "value":
            n_double,
    },
    {
        "metric":
            "double_review_rate_percent",
        "value":
            safe_percentage(
                n_double,
                n_items,
            ),
    },
    {
        "metric":
            "main_answer_label_valid_pairs",
        "value":
            n_answer_pairs,
    },
    {
        "metric":
            "main_question_label_valid_pairs",
        "value":
            n_question_pairs,
    },
    {
        "metric":
            "strict_complete_answer_pairs",
        "value":
            n_strict_answer,
    },
    {
        "metric":
            "answer_pairs_lost_if_strict_completeness_used",
        "value":
            n_answer_pairs
            - n_strict_answer,
    },
    {
        "metric":
            "sampling_total_variation_distance",
        "value":
            sampling_tvd,
    },
    {
        "metric":
            "items_requiring_adjudication",
        "value":
            n_adjudicated,
    },
    {
        "metric":
            "adjudication_rate_percent",
        "value":
            safe_percentage(
                n_adjudicated,
                n_items,
            ),
    },
    {
        "metric":
            "completed_adjudications",
        "value":
            n_resolved,
    },
    {
        "metric":
            "accept",
        "value":
            n_accept,
    },
    {
        "metric":
            "accept_with_variants_final_decision",
        "value":
            n_accept_variants,
    },
    {
        "metric":
            "corrected_and_accept",
        "value":
            n_corrected,
    },
    {
        "metric":
            "rewrite_question",
        "value":
            n_rewrite,
    },
    {
        "metric":
            "reject_item",
        "value":
            n_reject,
    },
    {
        "metric":
            "overall_rejection_rate_percent",
        "value":
            safe_percentage(
                n_reject,
                n_items,
            ),
    },
    {
        "metric":
            "overall_correction_rate_percent",
        "value":
            safe_percentage(
                n_corrected,
                n_items,
            ),
    },
    {
        "metric":
            "final_currently_retained_items",
        "value":
            len(
                final_retained_items
            ),
    },
    {
        "metric":
            "final_current_retention_rate_percent",
        "value":
            safe_percentage(
                len(
                    final_retained_items
                ),
                n_items,
            ),
    },
])


# ======================================================================
# PAPER-READY VALUES
# ======================================================================

def get_metric(
    sample,
    dimension,
    metric,
    field="estimate",
):

    selected = reliability[
        (
            reliability["sample"]
            == sample
        )
        &
        (
            reliability["dimension"]
            == dimension
        )
        &
        (
            reliability["metric"]
            == metric
        )
    ]

    if selected.empty:
        return np.nan

    return float(
        selected.iloc[0][field]
    )


ans_raw = get_metric(
    "main_label_valid",
    "answer_correctness",
    "raw_agreement",
)

ans_raw_low = get_metric(
    "main_label_valid",
    "answer_correctness",
    "raw_agreement",
    "ci95_low",
)

ans_raw_high = get_metric(
    "main_label_valid",
    "answer_correctness",
    "raw_agreement",
    "ci95_high",
)

ans_fleiss = get_metric(
    "main_label_valid",
    "answer_correctness",
    "fleiss_kappa",
)

ans_alpha = get_metric(
    "main_label_valid",
    "answer_correctness",
    "krippendorff_alpha_nominal",
)

ans_ac1 = get_metric(
    "main_label_valid",
    "answer_correctness",
    "gwet_ac1",
)

ans_linear = get_metric(
    "main_label_valid",
    "answer_correctness",
    "pooled_role_cohen_kappa_linear",
)


q_raw = get_metric(
    "main_label_valid",
    "question_quality",
    "raw_agreement",
)

q_fleiss = get_metric(
    "main_label_valid",
    "question_quality",
    "fleiss_kappa",
)

q_alpha = get_metric(
    "main_label_valid",
    "question_quality",
    "krippendorff_alpha_nominal",
)

q_ac1 = get_metric(
    "main_label_valid",
    "question_quality",
    "gwet_ac1",
)


variant_items = int(
    variant_detail[
        "merged_has_variants"
    ].sum()
)

total_alternative_variants = int(
    variant_detail[
        "merged_variant_count"
    ].sum()
)


paper_summary = f"""
SHORT-ANSWER VALIDATION — FINAL SCIENTIFIC SUMMARY
==================================================

DATASET AND REVIEW DESIGN
-------------------------
Total short-answer items: {n_items}

Double-reviewed items:
{n_double} / {n_items}
({safe_percentage(n_double, n_items):.2f}%)

Main answer-correctness IRR sample:
{n_answer_pairs} label-valid independent pairs.

Main question-quality IRR sample:
{n_question_pairs} label-valid independent pairs.

Strict-completeness sensitivity sample:
{n_strict_answer} answer-correctness pairs.

Therefore, using strict annotation completeness instead of label validity
would discard {n_answer_pairs - n_strict_answer} otherwise usable
double-review pairs from the main reliability analysis.


ANSWER CORRECTNESS
------------------
Raw exact agreement:
{100 * ans_raw:.2f}%

95% paired-bootstrap CI:
[{100 * ans_raw_low:.2f}%, {100 * ans_raw_high:.2f}%]

Fleiss' kappa:
{fmt(ans_fleiss)}

Krippendorff's alpha (nominal):
{fmt(ans_alpha)}

Gwet's AC1:
{fmt(ans_ac1)}

Supplementary pooled primary-vs-second linear-weighted Cohen's kappa:
{fmt(ans_linear)}


QUESTION QUALITY
----------------
Raw exact agreement:
{100 * q_raw:.2f}%

Fleiss' kappa:
{fmt(q_fleiss)}

Krippendorff's alpha:
{fmt(q_alpha)}

Gwet's AC1:
{fmt(q_ac1)}


ACCEPTED-ANSWER ENRICHMENT
--------------------------
Items enriched with at least one merged accepted variant:
{variant_items} / {n_items}
({safe_percentage(variant_items, n_items):.2f}%)

Total alternative accepted variants recorded:
{total_alternative_variants}

Mean acceptable-answer inventory size
(reference answer + accepted alternatives):
{variant_detail["acceptable_answer_inventory_size"].mean():.3f}

This statistic is independent from the adjudicator's final_decision label.
An item may have final_decision='accept' while still retaining multiple
accepted answer variants from the cross-validation stage.


SECOND-REVIEW REPRESENTATIVENESS
--------------------------------
Axe × Difficulty total variation distance:
{sampling_tvd:.4f}

0 means identical distributions. The closer this value is to zero,
the more closely the second-review sample preserves the composition of
the complete short-answer dataset.


ADJUDICATION
------------
Items escalated:
{n_adjudicated} / {n_items}
({safe_percentage(n_adjudicated, n_items):.2f}%)

Resolved:
{n_resolved} / {n_adjudicated}
({safe_percentage(n_resolved, n_adjudicated):.2f}%)

Final adjudication outcomes:

accept:
{n_accept}

accept_with_variants:
{n_accept_variants}

corrected_and_accept:
{n_corrected}

rewrite_question:
{n_rewrite}

reject_item:
{n_reject}

Overall rejection rate:
{safe_percentage(n_reject, n_items):.2f}%

Overall corrected-and-accepted rate:
{safe_percentage(n_corrected, n_items):.2f}%

Currently retained final items, excluding rejected and still-to-be-rewritten
questions:
{len(final_retained_items)} / {n_items}
({safe_percentage(len(final_retained_items), n_items):.2f}%)
""".strip()


# ======================================================================
# WRITE EXCEL
# ======================================================================

tables = {
    "OVERVIEW":
        overview,

    "IRR_ALL":
        reliability,

    "IRR_MAIN_ONLY":
        reliability[
            reliability["sample"]
            == "main_label_valid"
        ],

    "IRR_STRICT_SENSITIVITY":
        reliability[
            reliability["sample"]
            == "strict_complete_sensitivity"
        ],

    "ANSWER_CONFUSION":
        answer_confusion,

    "QUESTION_CONFUSION":
        question_confusion,

    "LABEL_MARGINALS":
        marginals,

    "SAMPLING_REP":
        sampling_rep,

    "VARIANT_SUMMARY":
        variant_summary,

    "VARIANT_ITEM_DETAIL":
        variant_detail,

    "DOUBLE_VARIANT_DETAIL":
        double_variant_detail,

    "ADJ_FINAL_DECISIONS":
        final_decision_distribution,

    "ADJ_TRANSITIONS":
        transition_matrix,

    "FINAL_RETAINED_ITEMS":
        final_retained_items,

    "DATA_QUALITY":
        quality_issues,
}


with pd.ExcelWriter(
    OUTPUT_XLSX,
    engine="openpyxl",
) as writer:

    for name, dataframe in tables.items():

        dataframe.to_excel(
            writer,
            sheet_name=name[:31],
            index=False,
        )


# ======================================================================
# FORMAT EXCEL
# ======================================================================

wb = load_workbook(
    OUTPUT_XLSX
)

for ws in wb.worksheets:

    ws.freeze_panes = "A2"

    if (
        ws.max_row >= 1
        and ws.max_column >= 1
    ):
        ws.auto_filter.ref = (
            ws.dimensions
        )

    for cell in ws[1]:
        cell.font = Font(
            bold=True
        )
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

    for row in ws.iter_rows(
        min_row=2
    ):
        for cell in row:
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True,
            )

    for col_idx in range(
        1,
        ws.max_column + 1,
    ):

        max_len = 10

        for row_idx in range(
            1,
            min(
                ws.max_row,
                300,
            ) + 1,
        ):

            value = ws.cell(
                row_idx,
                col_idx,
            ).value

            if value is not None:
                max_len = max(
                    max_len,
                    len(str(value)),
                )

        ws.column_dimensions[
            get_column_letter(
                col_idx
            )
        ].width = min(
            max_len + 2,
            55,
        )


wb.save(
    OUTPUT_XLSX
)

OUTPUT_TXT.write_text(
    paper_summary,
    encoding="utf-8",
)


# ======================================================================
# NOTEBOOK OUTPUT
# ======================================================================

print("=" * 90)
print("SHORT-ANSWER VALIDATION — FINAL SCIENTIFIC ANALYSIS")
print("=" * 90)

print()
print(f"Total items:                         {n_items}")
print(f"Double-reviewed items:               {n_double}")
print(
    f"Double-review rate:                  "
    f"{safe_percentage(n_double, n_items):.2f}%"
)

print()
print("RELIABILITY SAMPLE SIZES")
print("------------------------")
print(
    f"Answer label-valid pairs:            "
    f"{n_answer_pairs}"
)
print(
    f"Question label-valid pairs:          "
    f"{n_question_pairs}"
)
print(
    f"Strict-complete answer pairs:        "
    f"{n_strict_answer}"
)
print(
    f"Pairs lost under strict criterion:   "
    f"{n_answer_pairs - n_strict_answer}"
)

print()
print("MAIN RELIABILITY")
print("----------------")
print(
    reliability[
        reliability["sample"]
        == "main_label_valid"
    ].to_string(
        index=False
    )
)

print()
print("ACCEPTED VARIANTS")
print("-----------------")
print(
    variant_summary.to_string(
        index=False
    )
)

print()
print("FINAL ADJUDICATION DECISIONS")
print("----------------------------")
print(
    final_decision_distribution
    .to_string(
        index=False
    )
)

print()
print("DATA QUALITY")
print("------------")
print(
    quality_issues
    .groupby(
        [
            "severity",
            "issue_type",
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .to_string(
        index=False
    )
)

print()
print("=" * 90)
print("PAPER-READY SUMMARY")
print("=" * 90)
print()
print(paper_summary)

print()
print("=" * 90)
print("FILES CREATED")
print("=" * 90)
print(OUTPUT_XLSX.resolve())
print(OUTPUT_TXT.resolve())

SHORT-ANSWER VALIDATION — FINAL SCIENTIFIC ANALYSIS

Total items:                         296
Double-reviewed items:               59
Double-review rate:                  19.93%

RELIABILITY SAMPLE SIZES
------------------------
Answer label-valid pairs:            59
Question label-valid pairs:          59
Strict-complete answer pairs:        36
Pairs lost under strict criterion:   23

MAIN RELIABILITY
----------------
          sample          dimension  n_pairs                             metric  estimate  ci95_low  ci95_high
main_label_valid answer_correctness       59                      raw_agreement  0.830508  0.728814   0.915254
main_label_valid answer_correctness       59                       fleiss_kappa  0.300533 -0.083563   0.615277
main_label_valid answer_correctness       59         krippendorff_alpha_nominal  0.306461 -0.074334   0.631844
main_label_valid answer_correctness       59                           gwet_ac1  0.807142  0.677706   0.908776
main_label_valid answ